# DL-AG Reviewer Suite (3 seeds)

1. Runtime → Change runtime type → **GPU (T4)**
2. Upload `dlag_reviewer_colab.zip` to Colab Files **or** put it on Drive
3. Run cells top to bottom

Outputs: `multiseed_results_full/*.json`

In [ ]:
# === OPTION A: zip uploaded to Colab session storage ===
import os, zipfile
ZIP = "/content/dlag_reviewer_colab.zip"  # change if needed
assert os.path.exists(ZIP), f"Upload the zip first: {ZIP}"
!rm -rf /content/adversarial-glue
with zipfile.ZipFile(ZIP, "r") as z:
    z.extractall("/content")
# find extracted root
import pathlib
cands = list(pathlib.Path("/content").glob("**/train_ard.py"))
ROOT = str(cands[0].parent)
print("ROOT =", ROOT)
%cd {ROOT}
!nvidia-smi || true
import torch; print("cuda", torch.cuda.is_available())

In [ ]:
# === OPTION B (alternative): Drive mount ===
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/adversarial-glue

In [ ]:
!pip install -q -r requirements_dlag.txt scipy
import torch
print("cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# Sanity checks
from pathlib import Path
assert Path("sst2_augmented.csv").exists(), "missing sst2_augmented.csv"
assert Path("dev.json").exists(), "missing dev.json"
assert Path("dlag/layer1.py").exists(), "missing dlag package"
print("files OK")

In [ ]:
# FULL reviewer suite: DL-AG 3-seed + baselines + PKD + Table9 + t-tests
# Keep browser tab open; Colab free may disconnect — re-run skips finished seeds if checkpoints exist.
!python run_reviewer_suite.py --seeds 42 43 44 --max-train-pairs 15000 --dlag-epochs 3

In [ ]:
# Show results
from pathlib import Path
import json
res = Path("multiseed_results_full")
for p in sorted(res.glob("*.json")):
    print("\n===", p.name, "===")
    print(p.read_text(encoding="utf-8")[:2000])

In [ ]:
# Download results zip to your PC
!cd multiseed_results_full && zip -r /content/dlag_multiseed_results.zip .
from google.colab import files
files.download("/content/dlag_multiseed_results.zip")